In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=jKOVwY5Po3r91wiz1pjeMMBUIiLwHv&access_type=offline&code_challenge=CPI3PDaZIyZnFpVvnmfmFZMCNh4lmxsY7FzcPyL2_uA&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [2]:
!gcloud auth login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=abd5EXlGglhGfziyBg10QACBOnUZNI&access_type=offline&code_challenge=VZcDZTIBah2zLK1MuyrEg_39CKCc9OzNDjEAyhmMuaQ&code_challenge_method=S256


You are now logged in as [yt4@sanger.ac.uk].
Your current project is [open-targets-genetics-dev].  You can change this setting by running:
  $ gcloud config set project PROJECT_ID


In [3]:
import os

import hail as hl
import numpy as np
import pyspark.sql.functions as f
from pyspark.sql import DataFrame

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.dataset.study_locus import StudyLocus
from gentropy.susie_finemapper import SusieFineMapperStep

Loading BokehJS ...

In [4]:
hail_dir = os.path.dirname(hl.__file__)
session = Session(hail_home=hail_dir, start_hail=True, extended_spark_conf={
    "spark.driver.memory": "12g","spark.kryoserializer.buffer.max": "500m","spark.driver.maxResultSize":"2g",
    'spark.hadoop.fs.gs.requester.pays.buckets': 'requester-pays-bucket1,requester-pays-bucket2',
    'spark.hadoop.fs.gs.requester.pays.project.id': 'open-targets-genetics-dev',
    "spark.hadoop.fs.gs.requester.pays.mode":"AUTO"})

24/10/24 09:14:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
pip-installed Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jarRunning on Apache Spark version 3.3.4
SparkUI available at http://mib118093s:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.127-bb535cd096c5
LOGGING: writing to /dev/null


In [6]:
#x=session.spark.read.parquet("gs://gwas_catalog_sumstats_susie/study_index/")
x=StudyIndex.from_parquet(session,"gs://gwas_catalog_sumstats_susie/study_index/")
#x=StudyIndex.from_parquet(session,"gs://ot-team/dochoa/study_index_annotated")

In [7]:
x.df.count()

114252

In [11]:
x.df.groupBy("studyType").count().show()

+----------+-----+
| studyType|count|
+----------+-----+
|      gwas|95813|
|      pQTL|16736|
|Microbiome| 1703|
+----------+-----+



In [10]:
"pqtls" not in [
            "gwas",
            "pqtl",
        ]

True

In [15]:
study_index_df = x.filter(f.col("studyId") == "GCST90388627")

In [16]:
study_index_df.df.show(truncate=False)

+------------+---------+---------+--------------------------+------------------------+----------+------+---------------------+-----------+--------+---------------------------------------------------------------------------+----------------------+---------------+------------------+----------------------------------+--------------------+----------------------------------------------------------------------------+------+---------+--------+--------------+-------------------------------------------------------+---------------------------------------+------------------+---------------+-------------+--------------------+-----------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------+
|studyId     |projectId|studyType|traitFromSource           |traitFromSourceMappedIds|diseaseIds|geneId|biosampleFromSourceId|biosampleId|pubmedId|publicationTitle                                 

In [33]:
import logging
import time
from typing import Any

import numpy as np
import pandas as pd
import pyspark.sql.functions as f
import scipy as sc
from pyspark.sql import DataFrame, Row, Window
from pyspark.sql.functions import desc, row_number
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

from gentropy.common.session import Session
from gentropy.common.spark_helpers import (
    neglog_pvalue_to_mantissa_and_exponent,
    order_array_of_structs_by_field,
)
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus, StudyLocusQualityCheck
from gentropy.method.carma import CARMA
from gentropy.method.ld import LDAnnotator
from gentropy.method.ld_matrix_interface import LDMatrixInterface
from gentropy.method.sumstat_imputation import SummaryStatisticsImputation
from gentropy.method.susie_inf import SUSIE_inf

In [34]:
major_population = (
    x.df.select(
        "studyId",
        order_array_of_structs_by_field(
            "ldPopulationStructure", "relativeSampleSize"
        ).alias("ldPopulationStructure"),
    )
    .withColumn(
        "majorPopulation",
        f.when(
            f.col("ldPopulationStructure").isNotNull(),
            LDAnnotator._get_major_population(f.col("ldPopulationStructure")),
        ),
    )
)

In [35]:
y=x.df.join(major_population, on="studyId")

In [36]:
major_population.groupBy("majorPopulation").count().show()

+---------------+-----+
|majorPopulation|count|
+---------------+-----+
|           null|   18|
|            afr|13504|
|            nfe|86771|
|            eas|13172|
|            amr|  787|
+---------------+-----+



In [37]:
x.df.groupBy("studyType").count().show()

+----------+-----+
| studyType|count|
+----------+-----+
|      gwas|95813|
|      pQTL|16736|
|Microbiome| 1703|
+----------+-----+



In [18]:
keys_reasons = [
    "SMALL_NUMBER_OF_SNPS",
    "FAILED_GC_LAMBDA_CHECK",
    "FAILED_PZ_CHECK",
    "FAILED_MEAN_BETA_CHECK",
    "NO_OT_CURATION",
    "SUMSTATS_NOT_AVAILABLE",
]

qc_mappings_dict = StudyIndex.get_QC_mappings()
invalid_reasons = [
    qc_mappings_dict[key] for key in keys_reasons if key in qc_mappings_dict
]

In [19]:
invalid_reasons

['The number of SNPs in the study is below the expected threshold',
 'The GC lambda value is not within the expected range',
 'The PZ QC check values are not within the expected range',
 'The mean beta QC check value is not within the expected range',
 'GWAS Catalog study has not been curated by Open Targets',
 'Harmonized summary statistics are not available or empty']

In [45]:

y=y.withColumn(
                "FailedQC",
                f.arrays_overlap(
                    f.col("qualityControls"),
                    f.array([f.lit(reason) for reason in invalid_reasons]),
                ),
            )

In [39]:
y.show(truncate=False)

+------------+---------+---------+-------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+------+---------------------+-----------+--------+---------------------------------------------------------------------------------------------------------------------------------------------------

In [41]:
y.groupBy("qualityControls").count().show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|qualityControls                                                                                                                                                                                                                                |count|
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|[The mean beta QC check value is not within the expected range, The GC lambda value is not within the expected range, The number of SNPs in the study is below the expected threshold]                                                         |11   |
|[]     

In [47]:
y1=y.filter(f.col("FailedQC") == False)

In [48]:
y1.count()

46769

In [46]:
y.groupBy("FailedQC").count().show()

+--------+-----+
|FailedQC|count|
+--------+-----+
|    true|67483|
|   false|46769|
+--------+-----+



In [49]:
y1.groupBy("qualityControls").count().show(truncate=False)

+---------------+-----+
|qualityControls|count|
+---------------+-----+
|[]             |46769|
+---------------+-----+



In [24]:
y.filter((f.col("majorPopulation").isin(["nfe","eas","afr"])) & (f.col("hasSumstats")) & (f.col()) ).count()

113447

In [ ]:
a=session.spark.read.csv("gs://genetics-portal-dev-analysis/yt4/gwas_catalog_curation/20241004_output_curation.tsv", sep="\t", header=True)

In [ ]:
# Group by the exploded elements and count the occurrences
count_df = a.groupBy("analysisFlag").count()

# Show the results
count_df.show(truncate=False)

+---------------------+-----+
|analysisFlag         |count|
+---------------------+-----+
|Multivariate analysis|263  |
|ExWAS                |3991 |
|null                 |37293|
|Non-additive model   |88   |
|Metabolite           |15949|
|GxG                  |50   |
|GxE                  |140  |
|Case-case study      |77   |
+---------------------+-----+



In [ ]:
# Group by the exploded elements and count the occurrences
count_df = a.groupBy("isCurated").count()

# Show the results
count_df.show(truncate=False)

+---------+-----+
|isCurated|count|
+---------+-----+
|TRUE     |57851|
+---------+-----+



In [ ]:
# Group by the exploded elements and count the occurrences
count_df = a.groupBy("studyType").count()

# Show the results
count_df.show(truncate=False)

+----------+-----+
|studyType |count|
+----------+-----+
|pQTL      |16736|
|null      |39412|
|Microbiome|1703 |
+----------+-----+



In [5]:
x=StudyIndex.from_parquet(session,"gs://ot-team/dochoa/study_index_annotated")

In [3]:
x=StudyIndex.from_parquet(session,"gs://gwas_catalog_sumstats_susie/study_index/")

In [8]:
x=session.spark.read.parquet("gs://gwas_catalog_sumstats_susie/study_index/")

In [10]:
df=x
#x.df.count()


In [11]:
df=df.filter(f.col("studyId")=="GCST90027158")
df.count()

1

In [13]:
df.show(truncate=False)

+------------+---------+---------+--------+----------------------+---------------+------------------+------------------------------------------------------------------------------------+-------------------+-----------------------------------------------------------------------------------------------------------------------------+------------------------+----------------------------------+------------------------------------------------------------------------+--------------------+---------------------+------------------+------+---------+--------+---------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|studyId     |projectId|studyType|pubmedId|publicationFirstAuthor|publicationDate|publicationJournal|publicationTitle                                                                    |traitFromSource    |initialSampleSize                  

In [50]:
invalid_reasons = [
    "Multivariate analysis",
    "ExWAS",
    "Non-additive model",
    "GxG",
    "GxE",
    "Case-case study",
]
x_boolean =y.withColumn(
        "FailedQC1",
        f.arrays_overlap(
            f.col("analysisFlags"),
            f.array([f.lit(reason) for reason in invalid_reasons]),
        ),
    )

In [53]:
x_boolean.groupBy("FailedQC1").count().show()

+---------+------+
|FailedQC1| count|
+---------+------+
|     true|  4609|
|    false|109643|
+---------+------+



In [54]:
x_boolean.filter(f.col("FailedQC1") == False).groupBy("analysisFlags").count().show(truncate=False)

+-------------+-----+
|analysisFlags|count|
+-------------+-----+
|[]           |93694|
|[Metabolite] |15949|
+-------------+-----+



In [10]:
df=df.filter(f.col("hasSumstats")==True)
df.count()

52441

In [12]:
studyId="GCST90027158"

In [ ]:
from gentropy.method.ld import LDAnnotator
import numpy as np
import pandas as pd
import pyspark.sql.functions as f
import scipy as sc
from pyspark.sql import DataFrame, Row, Window
from pyspark.sql.functions import desc, row_number
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)
from gentropy.common.session import Session
from gentropy.common.spark_helpers import (
    neglog_pvalue_to_mantissa_and_exponent,
    order_array_of_structs_by_field,
)
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus, StudyLocusQualityCheck
from gentropy.method.carma import CARMA
from gentropy.method.ld import LDAnnotator
from gentropy.method.ld_matrix_interface import LDMatrixInterface
from gentropy.method.sumstat_imputation import SummaryStatisticsImputation
from gentropy.method.susie_inf import SUSIE_inf

In [ ]:
study_index_df = df.filter(f.col("studyId") == studyId)

In [ ]:
major_population = study_index_df.select(
    "studyId",
    order_array_of_structs_by_field(
        "ldPopulationStructure", "relativeSampleSize"
    )[0]["ldPopulation"].alias("majorPopulation"),
).collect()[0]["majorPopulation"]

In [ ]:
major_population

'nfe'

In [ ]:
study_index_df.show(truncate=False)

+------------+---------+---------+----------------------------------------------------------------------------+------------------------------+----------+------+---------------------+-----------+--------+------------------------------------------------------------------------------------------------------------+----------------------+---------------+------------------+----------------------------------+--------------------+----------------------------------------------------------------------+------+---------+--------+-------+---------------------+----------------+------------------+---------------+-----------------+--------------------+-----------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------+
|studyId     |projectId|studyType|traitFromSource                                                             |traitFromSourceMappedIds      |diseaseIds|geneId|biosampleFromSo

In [ ]:
major_population = (
    study_index_df.select(
        "studyId",
        order_array_of_structs_by_field(
            "ldPopulationStructure", "relativeSampleSize"
        ).alias("ldPopulationStructure"),
    )
    .withColumn(
        "majorPopulation",
        f.when(
            f.col("ldPopulationStructure").isNotNull(),
            LDAnnotator._get_major_population(f.col("ldPopulationStructure")),
        ),
    )
    .collect()[0]["majorPopulation"]
)

In [ ]:
major_population

'nfe'

In [ ]:
study_index_df.select("studyType").collect()[0]["studyType"]

'gwas'

In [ ]:
major_population not in ["nfe", "csa", "afr"]

False

In [ ]:
study_index_df.select("hasSumstats").collect()[0]["hasSumstats"]

True

In [ ]:
invalid_reasons = [
    "The PZ QC check values are not within the expected range.",
    "GWAS Catalog study has not been curated by Open Targets.",
    "The number of SNPs in the study is below the expected threshold.",
    "The mean beta QC check value is not within the expected range.",
    "The GC lambda value is not within the expected range.",
    "Harmonized summary statistics are not available or empty.",
]
x_boolean = (
    study_index_df.withColumn(
        "FailedQC",
        f.arrays_overlap(
            f.col("qualityControls"),
            f.array([f.lit(reason) for reason in invalid_reasons]),
        ),
    )
    .select("FailedQC")
    .collect()[0]["FailedQC"]
)
x_boolean

False

In [ ]:
study_index_df=study_index_df.drop("FailedQC")
invalid_reasons = [
    "Multivariate analysis",
    "ExWAS",
    "Non-additive model",
    "GxG",
    "GxE",
    "Case-case study",
]
x_boolean = (
    study_index_df.withColumn(
        "FailedQC",
        f.arrays_overlap(
            f.col("analysisFlags"),
            f.array([f.lit(reason) for reason in invalid_reasons]),
        ),
    )
    .select("FailedQC")
    .collect()[0]["FailedQC"]
)
x_boolean

True

In [ ]:
study_index_df.select("studyType").collect()[0]["studyType"] in [
            "gwas",
            "pqtl",
        ]

ConnectionRefusedError: [Errno 61] Connection refused

In [ ]:
exploded_df = df.withColumn("qualityControls_e", f.explode_outer("qualityControls"))

# Group by the exploded elements and count the occurrences
count_df = exploded_df.groupBy("qualityControls_e").count()

# Show the results
count_df.show(truncate=False)

In [ ]:
StudyIndex.get_QC_mappings()

# EGL

In [16]:
gsp=session.spark.read.csv("gs://genetics-portal-dev-analysis/yt4/GS_positives.txt",sep="\t",header=True)

In [17]:
gsp.show()

+--------------+-----+---------+---+---+---------------+------------------+-------------+------+-----------------+--------------------+--------------------+------+------------+-----------------+
|      study_id|chrom|      pos|ref|alt|        gene_id|proteinAttenuation|gs_confidence|gs_set|gs_confidence_num|gold_standard_status|      trait_reported|source|has_sumstats|chrom_pos_ref_alt|
+--------------+-----+---------+---+---+---------------+------------------+-------------+------+-----------------+--------------------+--------------------+------+------------+-----------------+
| GCST002442_19|    5|177415473|  T|  C|ENSG00000131187|       0.260109636|         High|ProGeM|                1|                   1|Blood metabolite ...|  GCST|       FALSE|  5_177415473_T_C|
|GCST002443_128|   19| 12899706|  A|  G|ENSG00000105607|       0.116959018|         High|ProGeM|                1|                   1|Blood metabolite ...|  GCST|       FALSE|  19_12899706_A_G|
|  GCST002364_9|   19| 32

In [18]:
si=session.spark.read.parquet("gs://genetics-portal-dev-data/22.09.1/outputs/lut/study-index")
si=si.drop("trait_reported")

In [19]:
si.printSchema()

root
 |-- study_id: string (nullable = true)
 |-- ancestry_initial: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ancestry_replication: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- n_cases: long (nullable = true)
 |-- n_initial: long (nullable = true)
 |-- n_replication: long (nullable = true)
 |-- pmid: string (nullable = true)
 |-- pub_author: string (nullable = true)
 |-- pub_date: string (nullable = true)
 |-- pub_journal: string (nullable = true)
 |-- pub_title: string (nullable = true)
 |-- has_sumstats: boolean (nullable = true)
 |-- num_assoc_loci: long (nullable = true)
 |-- source: string (nullable = true)
 |-- trait_efos: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- trait_category: string (nullable = true)



In [20]:
gsp=gsp.join(si, on="study_id", how="inner")

In [21]:
gsp.printSchema()

root
 |-- study_id: string (nullable = true)
 |-- chrom: string (nullable = true)
 |-- pos: string (nullable = true)
 |-- ref: string (nullable = true)
 |-- alt: string (nullable = true)
 |-- gene_id: string (nullable = true)
 |-- proteinAttenuation: string (nullable = true)
 |-- gs_confidence: string (nullable = true)
 |-- gs_set: string (nullable = true)
 |-- gs_confidence_num: string (nullable = true)
 |-- gold_standard_status: string (nullable = true)
 |-- trait_reported: string (nullable = true)
 |-- source: string (nullable = true)
 |-- has_sumstats: string (nullable = true)
 |-- chrom_pos_ref_alt: string (nullable = true)
 |-- ancestry_initial: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ancestry_replication: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- n_cases: long (nullable = true)
 |-- n_initial: long (nullable = true)
 |-- n_replication: long (nullable = true)
 |-- pmid: string (nullable = true)
 |-- pub_autho

In [22]:
gsp=gsp.select("trait_efos","study_id","gene_id","trait_reported")

In [23]:
gsp.show()

+-------------+-----------------+---------------+--------------------+
|   trait_efos|         study_id|        gene_id|      trait_reported|
+-------------+-----------------+---------------+--------------------+
|[EFO_0004576]|       GCST000150|ENSG00000119866|Fetal hemoglobin ...|
|[EFO_0004737]|     GCST000324_3|ENSG00000135697|Carotenoid and to...|
|[EFO_0004698]|      NEALE2_1200|ENSG00000142319|Sleeplessness / i...|
|[EFO_0008111]|      NEALE2_1309|ENSG00000243896|  Fresh fruit intake|
|[EFO_0008111]|      NEALE2_1319|ENSG00000169208|  Dried fruit intake|
|[EFO_0008111]|    NEALE2_1418_6|ENSG00000115850|Never/rarely have...|
|[EFO_0004330]|      NEALE2_1498|ENSG00000128271|       Coffee intake|
|[EFO_0004330]|      NEALE2_1498|ENSG00000140505|       Coffee intake|
|[EFO_0004330]|      NEALE2_1498|ENSG00000106546|       Coffee intake|
|[EFO_0007878]|      NEALE2_1568|ENSG00000257138|Average weekly re...|
|[EFO_0003784]|      NEALE2_1717|ENSG00000077498|         Skin colour|
|[EFO_

In [29]:
gsp.printSchema()

root
 |-- trait_efos: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- study_id: string (nullable = true)
 |-- gene_id: string (nullable = true)
 |-- trait_reported: string (nullable = true)



In [30]:
filtered_gsp = gsp.filter(f.size(f.col("trait_efos")) > 0)

In [31]:
filtered_gsp.count()

440

In [32]:
filtered_gsp.select("trait_efos","gene_id").distinct().count()

405

In [33]:
filtered_gsp=filtered_gsp.drop_duplicates(["trait_efos","gene_id"])

In [34]:
filtered_gsp.count()

405

In [35]:
filtered_gsp.show()

+-------------+--------------------+---------------+--------------------+
|   trait_efos|            study_id|        gene_id|      trait_reported|
+-------------+--------------------+---------------+--------------------+
| [CHEBI_9150]|NEALE2_20003_1140...|ENSG00000130203|Simvastatin | tre...|
|[EFO_0000095]|          GCST003468|ENSG00000137267|Chronic lymphocyt...|
|[EFO_0000095]|          GCST002073|ENSG00000171791|Chronic lymphocyt...|
|[EFO_0000180]|          GCST003183|ENSG00000160791|Setpoint viral lo...|
|[EFO_0000220]|          GCST005315|ENSG00000100804|Acute lymphoblast...|
|[EFO_0000275]|          GCST006414|ENSG00000055118| Atrial fibrillation|
|[EFO_0000275]|          GCST006414|ENSG00000113580| Atrial fibrillation|
|[EFO_0000275]|          GCST006061|ENSG00000123700| Atrial fibrillation|
|[EFO_0000275]|          GCST006414|ENSG00000126218| Atrial fibrillation|
|[EFO_0000275]|          GCST006414|ENSG00000138622| Atrial fibrillation|
|[EFO_0000275]|          GCST006061|EN

In [36]:
filtered_gsp.write.parquet("gs://genetics-portal-dev-analysis/yt4/20241024_gs_positives_405_with_efo")